In [1]:
import sunpy.map
import matplotlib.pyplot as plt
import numpy as np
import os
import re
from datetime import datetime
import scipy.ndimage as ndimage
import cv2
import Sunlimb
from matplotlib.colors import Normalize, LogNorm
import phasepack
from astropy.io import fits
import image as imag
from astropy.coordinates import SkyCoord
import astropy.units as u
from matplotlib.patches import Wedge
%matplotlib notebook

In [2]:
file_path = "E:/python/projects/alfven/data/corrected/10moving/"
files = [os.path.abspath(os.path.join(file_path, file)) for file in os.listdir(file_path) if file.endswith('.fits')]


def extract_datetime(file_name):
    match = re.search(r'\d{8}T\d{6}', file_name)
    if match:
        return datetime.strptime(match.group(), '%Y%m%dT%H%M%S')
    else:
        return datetime.min


files = sorted(files, key=extract_datetime)

In [3]:
def unsharp_mask(image, kernel_size=7):
    no_nan = np.nan_to_num(image.copy().astype(np.float64), nan=0)
    kernel = np.ones((kernel_size, kernel_size), np.float64) / (kernel_size * kernel_size)
    smoothed_image = cv2.filter2D(no_nan, -1, kernel)
    smoothed_image[np.isnan(image)] = np.nan
    return 2*image - smoothed_image

def make_st_data(files, deg_range, r, d_time, d_deg, save_fold=None, process=None):
    if not os.path.exists(save_fold):
        os.makedirs(save_fold)
    st_data = Sunlimb.space_time_plot(files=files, deg_range=deg_range, r=r, d_time=d_time, d_deg=d_deg, process=process, plot=False)
    M, m, ori, ft, PC, E0, T = phasepack.phasecong(st_data)
    save_name = os.path.join(save_fold, f'st_data_{r}Mm.csv')
    np.savetxt(save_name, PC[0]*st_data, delimiter=',')

In [4]:
deg_range = (182, 185)
d_deg = 0.01
d_time = 3
deg_vect = np.arange(deg_range[0], deg_range[1] + d_deg / 2, d_deg)
time_vect = np.arange(0, len(files) * d_time + 0.5 * d_time, d_time) / 60

In [5]:
from tqdm.notebook import tqdm
r_array = np.arange(14.5, 15.5, 0.5)
for r in tqdm(r_array, total=len(r_array), desc='making st_data'):
    make_st_data(files, deg_range, r, d_time, d_deg, './data/不同高度时空图/', unsharp_mask)

making st_data:   0%|          | 0/2 [00:00<?, ?it/s]

Processing files:   0%|          | 0/591 [00:00<?, ?it/s]

Processing files:   0%|          | 0/591 [00:00<?, ?it/s]